# [Experiment] Vanilla TabNet | Step LR | Higgs Boson

### Overview
This notebook establishes the fundamental performance baseline using the standard, unmodified TabNet architecture. The hyperparameters and structural configuration of this baseline have been meticulously aligned to follow the experimental setup detailed in the original TabNet paper as exactly as possible.

### Methodological Context: The Optimization Environment
To create a rigorously grounded control, this baseline employs a standard StepLR learning rate schedule. This specific schedule was selected because it mirrors the exact optimization strategy utilized by the original authors. This discrete, step-wise decay provides a historically validated, rigid optimization trajectory against which all of our subsequent architectural modifications will be measured.

### The Objective
The primary goal of this notebook is to capture the default performance metrics—specifically convergence rate, peak accuracy, and late-stage training stability—of the standard model as originally intended by its creators. This establishes the strict "ground truth" necessary to empirically evaluate the impact of introducing Kolmogorov-Arnold Network (KAN) layers.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Notes:
# - momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 37 epochs approximates the paper's 20k iterations
#   (based on a batch size of 16384 and ~8.8M training samples).
paper_config = {
    'n_d': 24,
    'n_a': 26,
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=0.02),
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=37, gamma=0.9),
    'mask_type': 'sparsemax'
}

clf_vanilla = TabNetClassifier(
    **paper_config,
    clip_value=2.,
    use_kan=False,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_vanilla.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.61869 | valid_accuracy: 0.68552 |  0:01:23s
epoch 25 | loss: 0.48174 | valid_accuracy: 0.76313 |  0:35:55s
epoch 50 | loss: 0.46681 | valid_accuracy: 0.77158 |  1:10:06s
epoch 75 | loss: 0.46348 | valid_accuracy: 0.77372 |  1:44:16s
epoch 100| loss: 0.46213 | valid_accuracy: 0.77402 |  2:18:21s
epoch 125| loss: 0.46088 | valid_accuracy: 0.77479 |  2:52:37s
epoch 150| loss: 0.46037 | valid_accuracy: 0.77611 |  3:26:55s
epoch 175| loss: 0.45962 | valid_accuracy: 0.77592 |  4:01:26s
epoch 200| loss: 0.45904 | valid_accuracy: 0.77672 |  4:35:57s
epoch 225| loss: 0.45849 | valid_accuracy: 0.77729 |  5:10:25s
epoch 250| loss: 0.45808 | valid_accuracy: 0.77742 |  5:44:53s
epoch 275| loss: 0.45765 | valid_accuracy: 0.7779  |  6:19:18s
epoch 300| loss: 0.45727 | valid_accuracy: 0.77765 |  6:53:41s
epoch 325| loss: 0.45709 | valid_accuracy: 0.77745 |  7:28:16s
epoch 350| loss: 0.45682 | valid_accuracy: 0.77824 |  8:03:05s
epoch 375| loss: 0.45662 | valid_accuracy: 0.77832 |  8

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_vanilla.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 76,680


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_vanilla.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.77939


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "Vanilla TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_vanilla.best_epoch,
    "training_history": clf_vanilla.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/01_vanilla_baseline_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_vanilla.save_model(f'{MODELS_DIR}/01_vanilla_baseline_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/01_vanilla_baseline_step_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/01_vanilla_baseline_step_lr_76k.zip
